In [4]:
import sys 
sys.path.append("../../Ray Tracing in One Weekend/")

from util.ray import Ray
from util.vec3 import Vec3
from util.color import write_color
from util.vec3 import unit_vector
from util.vec3 import dot
from tqdm import tqdm
import math


# HitRecord (updated)

In [2]:
class HitRecord:
    def __init__(self):
        self.p = None
        self.normal = None
        self.t = 0.0
        self.front_face = False

    def set_face_normal(self, ray, outward_normal):
        # dot(ray.direction, outward_normal) < 0 means ray is outside

        self.front_face = ray.direction.dot(outward_normal) < 0

        if self.front_face:
            self.normal = outward_normal
        else:
            self.normal = outward_normal * -1

# Hittable (Abstract Class)

In [3]:
from abc import ABC, abstractmethod

class Hittable(ABC):

    @abstractmethod
    def hit(self, ray, tmin, tmax, rec):
        pass

# Sphere (updated part only)

In [ ]:
class Sphere(Hittable):
    def __init__(self, center, radius):
        self.center = center
        self.radius = max(0.0, radius)

    def hit(self, ray, tmin, tmax, rec):

        oc = self.center - ray.origin

        a = ray.direction.length_squared()
        h = ray.direction.dot(oc)
        c = oc.length_squared() - self.radius * self.radius

        discriminant = h*h - a*c

        if discriminant < 0:
            return False

        sqrtd = math.sqrt(discriminant)

        root = (h - sqrtd) / a
        if root <= tmin or root >= tmax:
            root = (h + sqrtd) / a
            if root <= tmin or root >= tmax:
                return False

        rec.t = root
        rec.p = ray.at(rec.t)

        outward_normal = (rec.p - self.center) * (1.0 / self.radius)

        rec.set_face_normal(ray, outward_normal)

        return True

This is one of the most important ideas in ray tracing:
👉 **front face vs back face normals**

Let’s convert it into **Python + math + learning tables + clean logic** so it becomes intuitive.

---

# 1. The Core Problem (Why this exists)

A sphere has a normal:

$$
\text{normal} = \frac{P - C}{r}
$$

But there is a problem:

| Situation                | Normal direction                                  |
| ------------------------ | ------------------------------------------------- |
| Ray hits outside         | normal points opposite ray ✅                      |
| Ray starts inside sphere | normal points same direction as ray ❌ (confusing) |

So we must decide:

> Should normals always point outward OR always oppose ray?

We choose:

> ✔ store outward normal
> ✔ then fix it using ray direction

---

# 2. Key Idea (Simple rule)

We use dot product:

$$
\text{dot(ray dir, outward normal)}
$$

| Result | Meaning               |
| ------ | --------------------- |
| > 0    | ray is inside sphere  |
| < 0    | ray is outside sphere |

---

# 3. Final Design Decision (What book does)

We store:

| Variable       | Meaning                           |
| -------------- | --------------------------------- |
| outward_normal | geometric normal (always outward) |
| normal         | corrected normal (depends on ray) |
| front_face     | is ray hitting outside?           |

---

# 4. Python Implementation

## HitRecord (updated)

```python
class HitRecord:
    def __init__(self):
        self.p = None
        self.normal = None
        self.t = 0.0
        self.front_face = False

    def set_face_normal(self, ray, outward_normal):
        # dot(ray.direction, outward_normal) < 0 means ray is outside

        self.front_face = ray.direction.dot(outward_normal) < 0

        if self.front_face:
            self.normal = outward_normal
        else:
            self.normal = outward_normal * -1
```

---

## Sphere (updated part only)

```python
class Sphere(Hittable):
    def __init__(self, center, radius):
        self.center = center
        self.radius = max(0.0, radius)

    def hit(self, ray, tmin, tmax, rec):

        oc = self.center - ray.origin

        a = ray.direction.length_squared()
        h = ray.direction.dot(oc)
        c = oc.length_squared() - self.radius * self.radius

        discriminant = h*h - a*c

        if discriminant < 0:
            return False

        sqrtd = math.sqrt(discriminant)

        root = (h - sqrtd) / a
        if root <= tmin or root >= tmax:
            root = (h + sqrtd) / a
            if root <= tmin or root >= tmax:
                return False

        rec.t = root
        rec.p = ray.at(rec.t)

        outward_normal = (rec.p - self.center) * (1.0 / self.radius)

        rec.set_face_normal(ray, outward_normal)

        return True
```

---

# 5. Learning Tables

## A. Why dot product works

| Dot result | Meaning            |
| ---------- | ------------------ |
| positive   | same direction     |
| negative   | opposite direction |

So:

```text
ray inside sphere → dot > 0
ray outside sphere → dot < 0
```

---

## B. Geometry meaning

| Vector         | Meaning                   |
| -------------- | ------------------------- |
| ray.direction  | where ray is going        |
| outward_normal | surface outward direction |

Dot product tells:

> “Are they facing same way?”

---

## C. front_face meaning

| Value | Meaning                              |
| ----- | ------------------------------------ |
| True  | ray is outside → correct outward hit |
| False | ray is inside → flip normal          |

---

## D. Why we flip normal

We want:

> normal always oppose ray direction for shading consistency

So lighting works correctly:

* diffuse lighting depends on angle
* inside/outside must behave consistently

---

# 6. Visual intuition

## Case 1: Ray outside sphere

```
ray --->

      (sphere)
        ⟵ normal (outward)
```

dot(ray, normal) < 0
✔ front_face = True

---

## Case 2: Ray inside sphere

```
      ray --->
    (  ⟶ normal would be outward, but we flip it )
```

dot(ray, outward_normal) > 0
✔ front_face = False
✔ normal flipped inward

---

# 7. Why this design is powerful

Without this:

❌ lighting breaks for inside objects
❌ glass rendering becomes incorrect
❌ refraction direction becomes wrong

With this:

✔ same shading logic works everywhere
✔ glass, water, fog become possible
✔ materials become simple later

---

# 8. One-line summary

> We store outward geometry normal, then flip it using ray direction so shading always behaves consistently.

---

If you want next step, I can show:

👉 how this directly affects **Lambert shading (light calculation)**
👉 or how this becomes critical for **glass / refraction (Snell’s law)**
👉 or a full **step-by-step ray tracing execution trace** (ray → sphere → normal → color)
